## Hands on real clustering data - DESI DR1 LRG sample
In this session we will estimate (and interpret) the power spectrum of galaxy catalogs.
This is the first (compression) step of a standard clustering analysis; the second step consists in fitting these compressed measurements with a theory model to derive constraints on cosmological parameters.

### Installation

Computation will be much faster with the GPU backend.



In [ ]:
!pip install git+https://github.com/adematti/jax-power.git
!pip install git+https://github.com/cosmodesi/cosmoprimo#egg=cosmoprimo[camb]

#### Catalogs
Let's download DESI DR1 LRG catalogs available here: https://data.desi.lbl.gov/doc/releases/dr1/

In [ ]:
!wget https://data.desi.lbl.gov/public/dr1/survey/catalogs/dr1/LSS/iron/LSScats/v1.5/LRG_NGC_clustering.dat.fits
!wget https://data.desi.lbl.gov/public/dr1/survey/catalogs/dr1/LSS/iron/LSScats/v1.5/LRG_NGC_0_clustering.ran.fits

## Inspecting catalogs

In [ ]:
path_data = "LRG_NGC_clustering.dat.fits"
path_randoms = "LRG_NGC_0_clustering.ran.fits"

import numpy as np

from astropy.table import Table

data = Table.read(path_data)
randoms = Table.read(path_randoms)
print('Columns of the data (galaxy) catalog', list(data.columns))
print('Columns of the catalog of randoms', list(randoms.columns))

### RA/Dec distribution

In [ ]:
# Make a scatter plot of data RA/Dec and a histogram of data redshifts
# Check that weighted randoms have the same angular (RA/Dec) and redshift (Z) distribution as the data
from matplotlib import pyplot as plt
# Tip: for the RA/Dec plot, downsample the data and randoms for faster plots:
rng = np.random.RandomState(seed=42)
mask_data = rng.uniform(0., 1., len(data)) < 0.1
mask_randoms = rng.uniform(0., 1., len(randoms)) < mask_data.sum() / len(randoms)
# Then, e.g. plt.scatter(data['RA'][mask_data], data['DEC'][mask_data], s=1, label='data')
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.scatter(randoms['RA'][mask_randoms], randoms['DEC'][mask_randoms], s=1, label='randoms', color='C0')
ax.scatter(data['RA'][mask_data], data['DEC'][mask_data], s=1, label='data', color='C1')
ax.legend(frameon=False)
ax.set_xlabel('R.A. [deg]')
ax.set_ylabel('Dec. [deg]')
plt.show()

### Redshift ($z$) distribution

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.hist(data['Z'], histtype='step', density=True, label='data', color='C1')
ax.hist(randoms['Z'], histtype='step', density=True, label='randoms', color='C0')
ax.legend(frameon=False)
ax.set_xlabel('$z$')
plt.show()

# In the following, we will select a subsample of data, 0.4 < z < 0.6
zlim = (0.4, 0.6)
mask_zdata = (data['Z'] > zlim[0]) & (data['Z'] < zlim[1])
mask_zrandoms = (randoms['Z'] > zlim[0]) & (randoms['Z'] < zlim[1])

Let's first transform redshifts $z$ into distances, assuming a fiducial cosmology. Take as a fiducial cosmology the best fit cosmology of Planck2018 (TT, TE, EE, lowl, lensing).

In [ ]:
from cosmoprimo.fiducial import DESI
cosmo_fid = DESI(engine='camb')  # fiducial cosmology

def get_cartesian_positions(ra, dec, z):
    # Compute distance d
    d = cosmo_fid.comoving_radial_distance(z)
    # Turn distance d, RA (\phi), Dec (\pi/2-\theta) (mind degree -> radians!) into x, y, z Cartesian positions
    conv = np.pi / 180.
    theta, phi = dec * conv, ra * conv
    # TODO: Fill in x, y, z positions:
    # x = ...
    # y = ...
    # z = ...
    return np.column_stack([x, y, z])

data_positions = get_cartesian_positions(data['RA'], data['DEC'], data['Z'])
# Same for randoms
randoms_positions = get_cartesian_positions(randoms['RA'], randoms['DEC'], randoms['Z'])

For fun, make a 'wedge plot' of the data: a Cartesian 2D (x, y) slice between 1 and 2 deg in Dec. Do you see structures, filaments, voids?

In [ ]:
lim_dec = (1., 2.)
mask_data = (data['DEC'] > lim_dec[0]) & (data['DEC'] < lim_dec[1])
mask_randoms = (randoms['DEC'] > lim_dec[0]) & (randoms['DEC'] < lim_dec[1])

fig, ax = plt.subplots(1, 1, figsize=(10,) * 2)
ax.set_aspect('equal')
# Plot (-x, y) (minus sign just to orient the figure)
ax.scatter(- randoms_positions[mask_randoms, 0], randoms_positions[mask_randoms, 1], marker='.', s=40., alpha=0.01, color='C0')
ax.scatter(- data_positions[mask_data, 0], data_positions[mask_data, 1], marker='.', s=0.3, alpha=0.8, color='C1')
ax.tick_params(bottom=False, labelbottom=False, left=False, labelleft=False)
plt.show()

Galaxies in the data and randoms catalogs receive weights, to correct for observational systematic effects, such that the ensemble average of galaxy density (= "survey selection function") and that of randoms match:
- WEIGHT_SYS: weights to correct for photometric systematics: what are they? **Answer**: non-cosmological variations of target density (due to e.g. galactic dust, varying atmospheric, sky conditions)
- WEIGHT_COMP: weights to correct for fiber collisions: what are they? **Answer**: all galaxies do not receive a spectroscopic fibers (due to the limited density of fibers)
- WEIGHT_ZFAIL: weights to correct for redshift failures: what are they? **Answer**: reliable redshifts are not obtained for all galaxies (due to instrumental and observational conditions)

The total (completeness) weight is (up to some renormalization): WEIGHT = WEIGHT_SYS * WEIGHT_COMP * WEIGHT_ZFAIL.

In addition to completeness weights above, when computing 2pt statistics (correlation function or power spectrum), one can apply weights to minimize its variance: WEIGHT_FKP = 1/(1 + NZ * P0), with P0 the typical value of the power spectrum at the scales of interest, e.g. $10000 \; (\mathrm{Mpc}/h)^{3}$ (NZ is in $(\mathrm{Mpc}/h)^{-3}$).
See e.g. https://arxiv.org/pdf/astro-ph/9304022.pdf, eq. 2.3, for the variational demonstration (another, broader point-of-view is that of the optimal quadratic estimator, of which the FKP estimator we will use below is a simplification under some assumptions).

In [ ]:
# Compute data weights, WEIGHT * WEIGHT_FKP
data_weights = data['WEIGHT'] * data['WEIGHT_FKP']
# Same for randoms
randoms_weights = randoms['WEIGHT'] * randoms['WEIGHT_FKP']

# Now let's apply the z-selection
data_positions, data_weights = data_positions[mask_zdata], data_weights[mask_zdata]
randoms_positions, randoms_weights = randoms_positions[mask_zrandoms], randoms_weights[mask_zrandoms]

## Power spectrum


Computing pair counts for correlation function estimation remains somewhat slow (still tractable for current surveys, DESI, Euclid). Essentially, pair counts correspond to a convolution of the galaxy density field --- which means they can be computed as a simple product of their Fourier space counterpart.

More importantly, theorists tend to prefer thinking in terms of the power spectrum, as different $k$-modes are initially (almost) uncorrelated and evolve independently in the linear regime (see Julien's course).
Let's compute the power spectrum monopole step-by-step, to show how this works.

In [ ]:
from jax import numpy as jnp
from jaxpower import (
    get_mesh_attrs,
    compute_mesh2_spectrum,
    ParticleField,
    FKPField,
    BinMesh2Spectrum,
    compute_fkp2_spectrum_normalization,
    compute_fkp2_spectrum_shotnoise
)

attrs = get_mesh_attrs(data_positions, randoms_positions, boxpad=1.2, cellsize=15., dtype='c16')
print(attrs)  # Mesh attributes

# Define FKP field = data - randoms
data = ParticleField(data_positions, data_weights, attrs=attrs)
randoms = ParticleField(randoms_positions, randoms_weights, attrs=attrs)
fkp = FKPField(data, randoms)

In [ ]:
# Let's compute the power spectrum monopole "by hand"

# Paint FKP = (n_g - \bar{n}) * V_\mathrm{cell} field to mesh, with Triangular Shaped Cloud assignment scheme
# "compensate = True" to compensate for the corresponding smoothing
# interlacing to remove residual aliasing
mesh = fkp.paint(resampler='tsc', interlacing=2, compensate=True, out='complex')

# P(kvec) ~ |F(kvec)|^2
power = mesh * mesh.conj()
knorm = sum(kk**2 for kk in power.coords(sparse=True))**0.5

# Now bin the power spectrum: P(k) = mean_in_kbin(P(kvec))
edges = np.arange(0., mesh.attrs.knyq[0], 0.01)
ibin = jnp.digitize(knorm, edges, right=False) - 1
nmodes = jnp.bincount(ibin.ravel(), minlength=len(edges) - 1)  # number of modes that fall in the bin
kavg = jnp.bincount(ibin.ravel(), weights=knorm.ravel(), minlength=len(edges) - 1) / nmodes  # mean k of the bin
num_raw = jnp.bincount(ibin.ravel(), weights=power.real.ravel(), minlength=len(edges) - 1) / nmodes  # mean power of the bin

In [ ]:
fig, ax = plt.subplots()
ax.plot(kavg, kavg * num_raw, color='C0')
ax.set_xlabel(r'$k$ [$\mathrm{Mpc}/h$]')
plt.show()

We're almost there! We still have to:
- properly normalize the $P(k)$ measurement ($A = \sum V_\mathrm{cell} \bar{n}^2$)
- remove the shotnoise

In [ ]:
mesh_data = data.paint(resampler='cic', compensate=False, out='real')
mesh_randoms = randoms.paint(resampler='cic', compensate=False, out='real')
# Note: mesh_data, mesh_randoms are ~ numbers of galaxies per cell (not densities)
# So we have A = mesh_data * mesh_randoms (renormalized) / dV^2 * dV with dV = cellsize.prod()
# mesh_data and mesh_randoms are ~uncorrelated
# We take the product mesh_data * mesh_randoms instead of e.g. mesh_data**2 or mesh_randoms**2
norm = jnp.sum(mesh_data.value * mesh_data.sum() / mesh_randoms.sum() * mesh_randoms.value).real / attrs.cellsize.prod()

In [ ]:
# Estimate shotnoise
num_shotnoise = jnp.sum(data.weights**2) + (data.weights.sum() / randoms.weights.sum())**2 * jnp.sum(randoms.weights**2)

# Remove shotnoise, normalize
power = (num_raw - num_shotnoise) / norm

In [ ]:
fig, ax = plt.subplots()
ax.plot(kavg, kavg * power, color='C0')
ax.set_xlabel(r'$k$ [$\mathrm{Mpc}/h$]')
ax.set_ylabel(r'$k P_{\ell}(k)$ [$(\mathrm{Mpc}/h)^{2}$]')
plt.show()
# Much better!

In [ ]:
# Now, if we want to compute the higher order multipoles, it is a bit trickier (we have go to compute spherical harmonics)
# Let's just use jaxwpower for this

# Define k-bin edges (dk = 0.01) and multipoles
bin = BinMesh2Spectrum(mesh.attrs, edges={'step': 0.01}, ells=(0, 2, 4))

# Compute P(k)
power = compute_mesh2_spectrum(mesh, bin=bin, los='firstpoint')
# Add norm and shotnoise attributes
power = power.clone(norm=norm, num_shotnoise=num_shotnoise)

In [ ]:
fig, ax = plt.subplots()
for ill, ell in enumerate(power.ells):
    ax.plot(power.k(ell), power.k(ell) * power.view(projs=ell), color=f'C{ill:d}', label=rf'$\ell = {ell:d}$')
ax.legend(frameon=False)
ax.set_xlabel(r'$k$ [$\mathrm{Mpc}/h$]')
ax.set_ylabel(r'$k P_{\ell}(k)$ [$(\mathrm{Mpc}/h)^{2}$]')
plt.show()